# Meerjaars-consistentie: mutaties dateren en ruis wegfilteren

Vier jaargangen (2022 t/m 2025) in plaats van twee. Het idee:

- Een **echte mutatie** (PV geplaatst, dakkapel gebouwd) springt in précies één
  jaarovergang omhoog en blijft daarna — het dak is blijvend anders.
- **Ruis** (omvalling, vocht, schaduw) springt willekeurig tussen de paren heen en weer.

Per pand berekenen we vier verschilscores: de drie opeenvolgende paren
(22→23, 23→24, 24→25) en het eindverschil (22→25). Detectie gebeurt op het
eindverschil (zoals in v2: contourmasker + drempel per omgevingstype); de paren
bepalen daarna twee dingen:

1. **Datering** — het paar met de hoogste score is het jaarvak van de mutatie.
2. **Consistentie** — de **sprongmarge**: hoe ver het hoogste paar uitsteekt boven het
   gemiddelde van de andere twee, als fractie van dat hoogste paar. Eén schone sprong
   geeft ~0,6–0,8; gelijkmatige ruis blijft onder de ~0,5 (elk paar heeft immers een
   ruisbodem). Ook "verschijnt-en-verdwijnt" (hoog in twee paren) scoort laag — goed,
   want dat is meestal geen blijvende mutatie.

We valideren de datering tegen de BAG (nieuwbouw met bouwjaar 2023/2024 hoort in het
bijpassende jaarvak te springen) en meten hoeveel de consistentie-eis de werkvoorraad
verkleint.

Vereist: `.\stappen\6-mutatiescan.ps1` (el_2022 + el_2025) én
`.\stappen\7-meerjaars-downloaden.ps1` (el_2023 + el_2024).

In [ ]:
import io, json, sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from shapely.geometry import shape

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'scripts'))
from common import laad_config
from pdok import fetch_bag_panden, make_session

cfg = laad_config(REPO / 'config.yaml')
DATA = REPO / cfg['paden']['data']
RES = cfg['luchtfoto']['resolutie']
PX = cfg['luchtfoto']['tegelgrootte']
STAP = PX * RES

GEBIED = [101500, 395000, 107000, 400000]          # heel Etten-Leur
JAREN = [2022, 2023, 2024, 2025]
MAPPEN = [DATA / f'el_{j}' for j in JAREN]
BAG_ALLE = DATA / 'bag' / 'el_panden_alle.geojson'
DREMPEL_PCT = 95        # percentiel (per omgevingstype) op het eindverschil 22->25
SPRONGMARGE_MIN = 0.5   # consistentie-eis: zie de intro (ruis blijft hieronder)
STEDELIJK_VANAF = 6
MARKEER = '#FFD400'
BLAUW, GRIJS, AMBER = '#2563EB', '#94A3B8', '#F59E0B'

for m in MAPPEN:
    assert (m / 'tiles.json').exists(), f'{m} ontbreekt — draai stap 6 en 7 (zie intro)'
indexen = [json.loads((m / 'tiles.json').read_text()) for m in MAPPEN]

if not BAG_ALLE.exists():
    feats = fetch_bag_panden(make_session(), cfg['bag']['wfs_url'], tuple(GEBIED),
                             alleen_in_gebruik=False)
    BAG_ALLE.write_text(json.dumps({'type': 'FeatureCollection', 'bbox_rd': GEBIED, 'features': feats}))
panden = json.loads(BAG_ALLE.read_text())['features']
nieuwbouw = [f for f in panden if 2022 <= (f['properties'].get('bouwjaar') or 0) <= 2025]
bag_verklaard_ids = {f['properties']['identificatie'] for f in panden
                     if (f['properties'].get('bouwjaar') or 0) >= 2022
                     or f['properties'].get('status') in
                     ('Verbouwing pand', 'Sloopvergunning verleend',
                      'Bouwvergunning verleend', 'Bouw gestart')}
gemeenschappelijk = set(indexen[0])
for idx in indexen[1:]:
    gemeenschappelijk &= set(idx)
print(f'{len(panden)} BAG-panden, {len(gemeenschappelijk)} tegels in alle {len(JAREN)} jaargangen')

## 1. Vier scores per pand

Zelfde recept als v2 (blur, normalisatie, verschuivings-tolerant, contourmasker),
maar nu voor vier jaarparen. Reken op 15–30 minuten voor de hele stad.

In [ ]:
GRID_X, GRID_Y = GEBIED[0], GEBIED[1]
def tegel_id_voor(x, y):
    return f't_{int((x - GRID_X) // STAP):04d}_{int((y - GRID_Y) // STAP):04d}'

def laad_genorm(pad):
    beeld = Image.open(pad).convert('L').filter(ImageFilter.GaussianBlur(1.5))
    a = np.asarray(beeld, dtype=np.float32)
    return (a - a.mean()) / (a.std() + 1e-6)

per_tegel = defaultdict(list)
for f in panden:
    geom = shape(f['geometry'])
    if geom.area < 25:
        continue
    per_tegel[tegel_id_voor(geom.centroid.x, geom.centroid.y)].append((f, geom))

PAREN = [(0, 1), (1, 2), (2, 3)]        # opeenvolgende jaarovergangen
EIND = (0, 3)                            # 2022 -> 2025
VERSCHUIVINGEN = [(dx, dy) for dx in (-8, 0, 8) for dy in (-8, 0, 8)]

def minverschil(a, b):
    mv = np.full_like(a, np.inf)
    for dx, dy in VERSCHUIVINGEN:
        np.minimum(mv, np.abs(np.roll(b, (dy, dx), axis=(0, 1)) - a), out=mv)
    return mv

resultaten = {}   # pid -> dict(paren=[p1,p2,p3], eind=…, stratum=…, f=…, geom=…)
from tqdm.auto import tqdm
for tid in tqdm(sorted(gemeenschappelijk & set(per_tegel)), desc='tegels scoren'):
    beelden = [laad_genorm(m / idx[tid]['image']) for m, idx in zip(MAPPEN, indexen)]
    kaarten = [minverschil(beelden[i], beelden[j]) for i, j in PAREN]
    kaarten.append(minverschil(beelden[EIND[0]], beelden[EIND[1]]))
    bbox = indexen[0][tid]['bbox']
    stratum = ('stedelijk' if indexen[0][tid].get('n_panden', 0) >= STEDELIJK_VANAF
               else 'buitengebied')
    for f, geom in per_tegel[tid]:
        gx0, gy0, gx1, gy1 = geom.bounds
        x0 = max(0, int((gx0 - bbox[0]) / RES)); x1 = min(PX, int((gx1 - bbox[0]) / RES))
        y0 = max(0, int((bbox[3] - gy1) / RES)); y1 = min(PX, int((bbox[3] - gy0) / RES))
        if x1 - x0 < 12 or y1 - y0 < 12:
            continue
        masker = Image.new('1', (x1 - x0, y1 - y0), 0)
        tekenaar = ImageDraw.Draw(masker)
        for poly in (geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]):
            punten = [((px - bbox[0]) / RES - x0, (bbox[3] - py) / RES - y0)
                      for px, py in poly.exterior.coords]
            tekenaar.polygon(punten, fill=1)
        binnen = np.asarray(masker, dtype=bool)
        if binnen.sum() < 100:
            binnen = np.ones_like(binnen)
        waarden_pand = [float(k[y0:y1, x0:x1][binnen].mean()) for k in kaarten]
        resultaten[f['properties']['identificatie']] = {
            'paren': waarden_pand[:3], 'eind': waarden_pand[3],
            'stratum': stratum, 'f': f, 'geom': geom,
        }
print(f'{len(resultaten)} panden gescoord op {len(PAREN)} jaarparen + eindverschil')

## 2. Detectie, datering en consistentie

In [ ]:
OVERGANGEN = ['2022→2023', '2023→2024', '2024→2025']
eind = np.array([r['eind'] for r in resultaten.values()])
strata = np.array([r['stratum'] for r in resultaten.values()])
drempel_per = {naam: float(np.percentile(eind[strata == naam], DREMPEL_PCT))
               for naam in ('stedelijk', 'buitengebied') if (strata == naam).any()}

for pid, r in resultaten.items():
    p = np.array(r['paren'])
    r['sprong'] = int(p.argmax())
    rest = (p.sum() - p.max()) / (len(p) - 1)
    r['sprongmarge'] = float((p.max() - rest) / (p.max() + 1e-6))
    r['datering'] = OVERGANGEN[r['sprong']]
    r['detectie'] = r['eind'] >= drempel_per[r['stratum']]
    r['consistent'] = r['sprongmarge'] >= SPRONGMARGE_MIN

detecties = {pid: r for pid, r in resultaten.items() if r['detectie']}
onverklaard = {pid: r for pid, r in detecties.items() if pid not in bag_verklaard_ids}
consistent_onverkl = {pid: r for pid, r in onverklaard.items() if r['consistent']}

bewaar_map = DATA / 'mutaties_preview'
bewaar_map.mkdir(exist_ok=True)
stappen = [('Panden geanalyseerd', len(resultaten), GRIJS),
           (f'Eindverschil ≥ P{DREMPEL_PCT} (per stratum)', len(detecties), BLAUW),
           ('Onverklaard door BAG', len(onverklaard), BLAUW),
           (f'... én consistent (sprongmarge ≥ {SPRONGMARGE_MIN}) → werkvoorraad',
            len(consistent_onverkl), AMBER)]
fig, ax = plt.subplots(figsize=(7.5, 2.8))
posities = range(len(stappen))
ax.barh(posities, [n for _, n, _ in stappen], height=0.55, color=[k for _, _, k in stappen])
ax.set_yticks(posities, [naam for naam, _, _ in stappen])
ax.invert_yaxis(); ax.set_xscale('log')
for i, (_, n, _) in enumerate(stappen):
    ax.text(n * 1.12, i, str(n), va='center', color='#555555', fontsize=9)
ax.set_title('Trechter mét consistentie-eis (logaritmische as)', loc='left', fontsize=11)
ax.spines[['top', 'right']].set_visible(False); ax.tick_params(length=0)
plt.tight_layout()
fig.savefig(bewaar_map / 'trechter_meerjaars.jpg', dpi=110, bbox_inches='tight')
plt.show()

if onverklaard:
    reductie = 1 - len(consistent_onverkl) / len(onverklaard)
    print(f'Consistentie-eis verkleint de onverklaarde werkvoorraad met {reductie:.0%} '
          f'({len(onverklaard)} → {len(consistent_onverkl)}).')
van_datering = np.array([r['datering'] for r in consistent_onverkl.values()])
for o in OVERGANGEN:
    print(f'  gedateerd {o}: {int((van_datering == o).sum())} mutaties')

## 3. Validatie van de datering tegen de BAG

Nieuwbouw met bouwjaar 2023 hoort te springen in 2022→2023 of 2023→2024 (de bouw is
vaak al zichtbaar vóór de oplevering); bouwjaar 2024 in 2023→2024 of 2024→2025.
Hoe vaak wijst het hoogste paar het juiste jaarvak aan?

In [ ]:
verwacht = {2023: {0, 1}, 2024: {1, 2}}
print(f'{"bouwjaar":<10}{"gescoord":>9}{"datering klopt":>16}')
kloppend_totaal = gescoord_totaal = 0
for jaar, ok_sprongen in verwacht.items():
    ids = [f['properties']['identificatie'] for f in nieuwbouw
           if f['properties']['bouwjaar'] == jaar]
    ids = [i for i in ids if i in resultaten]
    klopt = sum(1 for i in ids if resultaten[i]['sprong'] in ok_sprongen)
    kloppend_totaal += klopt; gescoord_totaal += len(ids)
    pct = klopt / len(ids) if ids else 0
    print(f'{jaar:<10}{len(ids):>9}{klopt:>10} ({pct:.0%})')
if gescoord_totaal:
    print(f'\nTotaal: {kloppend_totaal}/{gescoord_totaal} '
          f'({kloppend_totaal / gescoord_totaal:.0%}) correct gedateerd — dit is de\n'
          f'kwaliteitsmaat van de meerjaars-aanpak (puur kansniveau zou ~67% zijn,\n'
          f'omdat twee van de drie overgangen als \'goed\' tellen).')

## 4. Jaarreeks-galerijen

Vier panelen per pand (2022 | 2023 | 2024 | 2025) met de gele markering en de
afgeleide datering in de titel — eerst de consistente werkvoorraad, dan ter
vergelijking een paar niet-consistente detecties (vermoedelijk ruis).

In [ ]:
def jaarreeks(pid, r, assen_rij):
    geom = r['geom']
    tid = tegel_id_voor(geom.centroid.x, geom.centroid.y)
    bbox = indexen[0][tid]['bbox']
    gx0, gy0, gx1, gy1 = geom.bounds
    x0, x1 = (gx0 - bbox[0]) / RES, (gx1 - bbox[0]) / RES
    y0, y1 = (bbox[3] - gy1) / RES, (bbox[3] - gy0) / RES
    m = 14 / RES
    crop = (max(0, x0 - m), max(0, y0 - m), min(PX, x1 + m), min(PX, y1 + m))
    for ax, jaar, map_, idx in zip(assen_rij, JAREN, MAPPEN, indexen):
        beeld = Image.open(map_ / idx[tid]['image']).convert('RGB')
        ImageDraw.Draw(beeld).rectangle([x0, y0, x1, y1], outline=MARKEER, width=4)
        ax.imshow(beeld.crop(crop))
        ax.set_title(str(jaar), fontsize=9)
        ax.axis('off')

def toon_reeksen(items, titel, aantal=4):
    items = items[:aantal]
    if not items:
        print('(geen voorbeelden)'); return
    fig, assen = plt.subplots(len(items), 4, figsize=(12.5, 3.3 * len(items)), squeeze=False)
    for (pid, r), rij in zip(items, assen):
        jaarreeks(pid, r, rij)
        rij[0].set_ylabel(f"{r['datering']}\nmarge {r['sprongmarge']:.2f}", fontsize=8)
        rij[0].axis('on'); rij[0].set_xticks([]); rij[0].set_yticks([])
        for kant in rij[0].spines.values():
            kant.set_visible(False)
    fig.suptitle(titel, fontsize=12)
    plt.tight_layout(); plt.show()

top_consistent = sorted(consistent_onverkl.items(), key=lambda kv: -kv[1]['eind'])
toon_reeksen(top_consistent, 'Werkvoorraad: consistente onverklaarde mutaties, mét datering')
ruis = sorted(((pid, r) for pid, r in onverklaard.items() if not r['consistent']),
              key=lambda kv: -kv[1]['eind'])
toon_reeksen(ruis, 'Ter vergelijking: niet-consistente detecties (vermoedelijk ruis)', aantal=3)

## 5. Conclusie

In [ ]:
print(f'Meerjaars-consistentie over {len(JAREN)} jaargangen, drempel P{DREMPEL_PCT}, '
      f'sprongmarge-eis {SPRONGMARGE_MIN}:')
print(f'- {len(detecties)} detecties op eindverschil; {len(onverklaard)} onverklaard door BAG')
print(f'- Consistentie-eis: werkvoorraad {len(onverklaard)} → {len(consistent_onverkl)} '
      f'panden, elk met een jaarvak-datering')
print(f'- Dateringskwaliteit t.o.v. BAG-nieuwbouw: zie sectie 3')
print()
print('Afstellen: SPRONGMARGE_MIN hoger = strengere ruisfilter, maar panden met twéé echte')
print('veranderingen (bijv. nieuwbouw + later PV) vallen dan af — bekijk de niet-consistente')
print('galerij om te zien wat je wegfiltert vóór je aanscherpt. DREMPEL_PCT werkt zoals in')
print('de eerdere rapporten. Alleen de cellen vanaf sectie 2 opnieuw draaien is genoeg.')

---
*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*